# H2: official baseline vs full algorithm — seed 0

Paper references motivate baseline selection only. No comparison with paper accuracy numbers.
Pinned source commit; no ZIP overlay. Full registered horizon, final-epoch models, lightweight API/dataset preflight; no benchmark training in preflight; no truncation.
H1: CIFAR-100, BN ResNet32 PyCIL, 20 tasks ×5 classes, memory2000, 200/170 epochs.
H2: validation-only geometry grid selected from task2 within three eligible windows, then frozen; official LPR online Split-CIFAR100, SlimResNet18, 20 experiences, minibatch10 ×3 repeats, class-balanced replay2000, validation5%, LPR omega0=4/beta=2/update30 from official README, SGD0.1 without momentum/weight decay/scheduler.
H3-small: original Continual ImageNet32, first100 seed0 binary pairs, 50 epochs/task, batch100, ConvNet, SGD0.01 momentum0.9, native head reset. VNA at first full window of each task, then every200 updates/window128; about300 opportunities over100 tasks; at most16 exact swap candidates plus identity, selected by deterministic first-order LM ranking. Shortlist-only guarantee, not global optimization. Single current-task CE objective, no replay. CBP remains main baseline; BP, VNA-only and GIR-only are queued ablations. This pilot does not reproduce full-horizon CBP results.
H4: Rotated MNIST, 20 fixed angles 0,15,...,285 degrees; fixed 10-class head, ER vs ER + prior-domain constrained VNA; training-only domain IDs partition loss constraints, shared fixed backbone/head.

Inputs: optional CIFAR-100 Python train/test/meta; auto-download from official torchvision source if absent.
Enable Internet and Kaggle Secret `github_token`. Select T4 ×2; baseline GPU0, full GPU1, one BLAS thread per lane.
H2 builds an isolated Python3.10 environment for the pinned original Avalanche0.4 stack.
Optional previous Output is only for exact resume. H3-small queues five arms (GPU0 CBP then BP; GPU1 full then VNA-only then GIR-only),60000 native updates per arm plus bounded interventions; the full5000×250 config remains separate and is not queued. Exact T4 timings remain unmeasured. Other branches may also need multiple 11-hour sessions. The horizon is never shortened.


In [ ]:
BRANCH = 'H2'
REPO_COMMIT = 'c2398527f589dcd458288a75316fe5650ebb2db4'
RUN_TAG = 'standard_h2_s0_c2398527f589'
DATA_DIR = None  # optional explicit root under /kaggle/input
JOBS = ([{'name': 'lpr_s0', 'module': 'experiments.standard_online', 'args': ['--seed', '0', '--config', 'configs/standard_cl/h2_online_cifar100.json', '--arm', 'lpr']}], [{'name': 'geometry_vna_s0', 'module': 'experiments.standard_online', 'args': ['--seed', '0', '--config', 'configs/standard_cl/h2_online_cifar100.json', '--arm', 'h2']}])
import time
SESSION_START = time.time()


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/standard_cl")
OUTPUT = RUN_ROOT / "standard_outputs"

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)


In [ ]:
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION']='python'
print('KAGGLE HOST',sys.version,flush=True)
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout,flush=True)
import importlib.util
for package in ('pandas','matplotlib'):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
def fetch(name,url,pin):
    path=REPO/'third_party'/name
    if not path.exists():
        subprocess.run(['git','clone','--filter=blob:none','--no-checkout',url,str(path)],check=True)
        subprocess.run(['git','-C',str(path),'fetch','origin',pin],check=True)
        subprocess.run(['git','-C',str(path),'checkout','--detach',pin],check=True)
    if subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()!=pin:
        raise ValueError('Upstream pin mismatch: '+name)
if BRANCH=='H3':fetch('loss-of-plasticity','https://github.com/shibhansh/loss-of-plasticity.git','a6b79580d85f3025bdb601566d3627c5f489f13b')
else:fetch('PyCIL','https://github.com/LAMDA-CL/PyCIL.git','f3509b8ca3f20660ce4aa13f19d5283de81b4b35')
if BRANCH=='H2':fetch('LPR','https://github.com/plai-group/LPR.git','76c44f8f2481f4e23881b2463eb58aaf94affabb')
PYTHON=sys.executable
if BRANCH=='H2':
    if importlib.util.find_spec('uv') is None:subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    environment=RUN_ROOT/'lpr_env'
    if not (environment/'bin/python').exists():
        subprocess.run([sys.executable,'-m','uv','venv','--python','3.10',str(environment)],check=True)
    PYTHON=str(environment/'bin/python')
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PYTHON,'torch==2.2.2','torchvision==0.17.2',
        '-r',str(REPO/'requirements-h2.txt'),'-r',str(REPO/'requirements-upstream.txt'),'-e',str(REPO)],check=True)
    # Only this tested torch typing metadata exception is allowed by the API/dependency gate.
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PYTHON,'--no-deps',
        'avalanche-lib==0.4.0','typing-extensions==4.4.0'],check=True)
else:
    subprocess.run([PYTHON,'-m','pip','install','-q','-e',str(REPO)+'[vision,diagnostics]',
        '-r',str(REPO/'requirements-upstream.txt')],check=True)
if BRANCH=='H3':subprocess.run([PYTHON,'-m','pip','install','-q','gdown==5.2.0'],check=True)
sys.path.insert(0,str(REPO))
CONFIG={'H1':'configs/standard_cl/h1_cifar100_bn.json','H2':'configs/standard_cl/h2_online_cifar100.json',
        'H3':'configs/standard_cl/h3_imagenet_small.json','H4':'configs/standard_cl/h4_rotated_mnist.json'}[BRANCH]
PREFLIGHT_REPORT=RUN_ROOT/'preflight'/RUN_TAG/'preflight.json'
preflight=[PYTHON,'-m','diagnostics.standard_preflight','--branch',BRANCH,'--config',CONFIG,
    '--stage',str(RUN_ROOT/'input_data'),'--out',str(PREFLIGHT_REPORT),'--device','cuda:0']
if DATA_DIR:preflight+=['--data-dir',DATA_DIR]
env={**os.environ,'PYTHONPATH':str(REPO),'OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','MPLBACKEND':'Agg'}
subprocess.run(preflight,cwd=REPO,env=env,check=True)
report=json.loads(PREFLIGHT_REPORT.read_text())
if report['status']!='PASSED':raise RuntimeError('Preflight did not pass; training blocked')
RESOLVED_DATA=report['data_dir']
print('PREFLIGHT PASSED; seed0; input=',RESOLVED_DATA,'source=',REPO_COMMIT,flush=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives,archive_outputs
from experiments.vna_resume import launch_logged,print_heartbeat
from experiments.io_utils import atomic_json_save
from scripts.collect_standard_cl import collect,export_comparison
ROOT_OUT=OUTPUT/RUN_TAG
if any(ROOT_OUT.rglob('latest.pt')) or any(ROOT_OUT.rglob('summary.json')):
    print('Keeping local working checkpoints; attached older output will not overwrite them',flush=True)
else:restore_stage_archives(Path('/kaggle/input'),OUTPUT,'standard_outputs',prefix=RUN_TAG)
ROOT_OUT.mkdir(parents=True,exist_ok=True)
VNA_DEADLINE=SESSION_START+11*3600
base_env={**os.environ,'MPLBACKEND':'Agg','PYTHONPATH':str(REPO),'PYTHONUNBUFFERED':'1','TQDM_DISABLE':'1',
    'PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION':'python','OMP_NUM_THREADS':'1',
    'MKL_NUM_THREADS':'1','VNA_DEADLINE':str(VNA_DEADLINE)}
configs=RUN_ROOT/'effective_configs';configs.mkdir(exist_ok=True)
def command(job):
    args=list(job['args'])
    if BRANCH=='H1':
        index=args.index('--config') if '--config' in args else None
        source=args[index+1] if index is not None else 'configs/standard_cl/h1_cifar100_bn.json'
        config=json.loads((REPO/source).read_text());config['workers']=1
        target=configs/Path(source).name;target.write_text(json.dumps(config,indent=2)+'\n')
        if index is not None:args[index+1]=str(target)
        else:args+=['--config',str(target)]
    args+=['--data-dir',RESOLVED_DATA,'--device','cuda:0']
    return [PYTHON,'-m',job['module'],*args,'--out',str(ROOT_OUT/job['name'])]
commands=[[command(job) for job in lane] for lane in JOBS]
manifest=dict(branch=BRANCH,seed=0,source_commit=REPO_COMMIT,jobs=JOBS,commands=commands,
    paper_role='baseline choice rationale only; no paper-number reproduction gate',
    runtime_overrides={'workers':1 if BRANCH=='H1' else 0,'OMP_NUM_THREADS':1,'MKL_NUM_THREADS':1},
    preflight=str(PREFLIGHT_REPORT),horizons='registered H3-small pilot; full H1/H2/H4; exit75 resumes without changing identity')
atomic_json_save(manifest,ROOT_OUT/'notebook_protocol.json')
stop=threading.Event();paused=threading.Event();lanes={};archive_lock=threading.Lock();archive_errors=[]
def archive():
    with archive_lock:return archive_outputs(RUN_ROOT,folders=('standard_outputs','preflight'),exclude_parts=('data',))
def periodic():
    while not stop.wait(300):
        try:archive()
        except Exception as exc:archive_errors.append(repr(exc));print('ARCHIVE ERROR',repr(exc),flush=True)
def heartbeat():
    while not stop.wait(30):print_heartbeat(RUN_ROOT,SESSION_START,VNA_DEADLINE,lanes)
def lane(index):
    env={**base_env,'CUDA_VISIBLE_DEVICES':str(index)}
    for job,cmd in zip(JOBS[index],commands[index]):
        print('START',job['name'],cmd,flush=True)
        if not launch_logged(cmd,cwd=REPO,env=env,log_path=RUN_ROOT/(RUN_TAG+'_'+job['name']+'.log'),
            lane=index,job=job['name'],lanes=lanes,paused=paused,deadline=VNA_DEADLINE):break
        archive()
threads=[threading.Thread(target=periodic,daemon=True),threading.Thread(target=heartbeat,daemon=True)]
for thread in threads:thread.start()
try:
    for index in range(2):
        probe=f"from experiments.vna_resume import reset_cuda_peak_memory_stats; import torch; reset_cuda_peak_memory_stats('cuda:0'); print('FRESH LANE {index} CUDA:', torch.cuda.device_count(), torch.cuda.get_device_name(0), flush=True)"
        subprocess.run([PYTHON,'-c',probe],cwd=REPO,env={**base_env,'CUDA_VISIBLE_DEVICES':str(index)},check=True)
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures=[pool.submit(lane,index) for index in range(2)]
        for future in futures:future.result()
finally:
    stop.set()
    for thread in threads:thread.join(timeout=10)
    atomic_json_save(collect([ROOT_OUT]),ROOT_OUT/'collected.json')
    export_comparison(ROOT_OUT)
    atomic_json_save(dict(paused=paused.is_set(),archive_errors=archive_errors),ROOT_OUT/'session.json')
    archive()


In [ ]:
import pandas as pd
from IPython.display import Image,display
comparison=ROOT_OUT/'comparison.csv'
if comparison.exists():display(pd.read_csv(comparison))
if (ROOT_OUT/'accuracy_curves.png').exists():display(Image(filename=str(ROOT_OUT/'accuracy_curves.png')))
print('Archive:',RUN_ROOT.with_suffix('.tar.gz'))
print('If PAUSED: attach this Output and the same dataset, rerun the same source pin/run tag.')
print('No source ZIP, Gram authority or old VNA checkpoint is needed.')
